# Imports

In [3]:
%load_ext autoreload
%autoreload 2

In [2]:
from rdflib import Graph, Literal, RDF, URIRef, Namespace
from urllib.parse import quote, unquote
import pandas as pd
from datetime import timedelta


In [4]:
import sys  
sys.path.insert(1, './Business Process Optimization Competition 2023')
from simulator import Simulator
from problems import MinedProblem
sys.path.insert(1, './src')
from KGPlanner import KGPlanner 
from ProcessKnowledgeGraph import ProcessKnowledgeGraph
from SHACLAllocator import SHACLAllocator 
from utils import *

# Run

### Planner & Allocator Setup
The planner is the interface to the simulation frame, implementing a function to assign a set of open tasks.
The allocation actually manages the reasoning based on the encoded ontology and rules. In this example setup, we load additional knowledge as desribed in the paper's example use case. 

In [8]:
planner = KGPlanner()
planner.allocator.load_extension(rules_ext='./src/extension_sepOfConcerns.ttl')
planner.allocator.load_extension(rules_ext='./src/extension_seniority.ttl')
planner.allocator.load_extension(ontology_ext='./src/extension_ontology.ttl')

### Simulation Frame Setup
The simulation frame and simulated process are taken from the BPOC 2023. We adapt the simulation numbers as to allow for more interesting allocation situations

In [9]:
simulator = Simulator(planner=planner, instance_file="./Business Process Optimization Competition 2023/BPI Challenge 2017 - instance 2.pickle")
problem = simulator.problem
# Increase Arrival Rate so more also more resources are provided
problem.interarrival_time._scale *= 0.5
# Increase likelihood for assessing fraud, so separation of concerns actually triggers
problem.next_task_distribution['W_Validate application'] = list(map(lambda x : (x[0] * 0.8 + (0.2 if (x[1] == 'W_Assess potential fraud') else 0), x[1]), problem.next_task_distribution['W_Validate application']))

### Run the Simulation

In [41]:
result = simulator.run(1)
print(result)
# draw_graph(planner.graph)


case-4 Task: W_Complete application Available: {'User_44', 'User_21', 'User_17'} Allowed: {'User_44', 'User_21', 'User_17'}
Assigning: User_17 to task-29 considering the following: 
 Resource has relevant role(s) for activity 'W_Complete application'


case-8 Task: W_Complete application Available: {'User_44', 'User_21'} Allowed: {'User_44', 'User_21'}
Assigning: User_44 to task-19 considering the following: 
 Resource has relevant role(s) for activity 'W_Complete application'


case-14 Task: W_Complete application Available: {'User_21', 'User_60', 'User_120', 'User_23'} Allowed: {'User_21', 'User_60', 'User_23'}


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f0ff7a40ac0>>
Traceback (most recent call last):
  File "/mnt/c/unsynchronized/projects/.Graph Workbench/Workbench/knowledge-graph-resource-allocation/.venv/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


Assigning: User_23 to task-25 considering the following: 
 Resource has relevant role(s) for activity 'W_Complete application'


case-15 Task: W_Complete application Available: {'User_21', 'User_60', 'User_120'} Allowed: {'User_21', 'User_60'}
Assigning: User_21 to task-30 considering the following: 
 Resource has relevant role(s) for activity 'W_Complete application'

Assigning: User_60 to task-20 considering the following: 
 Resource has relevant role(s) for activity 'W_Complete application'

(0.2317017182936315, 'COMPLETED: you completed 1 hours of simulated customer cases. 16 cases started. 1 cases run to completion.')
